# Understanding the Business Problem

Education and cultural infrastructure is a very important factor when assessing the quality of the London borough. Although London is one of the main epicentres for education and culture in Europe and the World, it's crucial to consier the size of the London and the fact that some boroughs might be overlooked when discussing schools or places like museums, theatres, or libraries.

This project aims to showcase the educational institutions and cultural infrastructure in Tower Hamlet borough. Many people might be unaware of the amount of the opportunities presented in the borough. Visualising all the schools (primary to secondary schools) and all the cultural amentities, the main goal of this project is to make people, especially Tower Hamlet locals, as well as the general public in London aware of what Tower Hamlets has to offer.

The project might support individuals and families in choosing best places to live in Tower Hamlets based on the proximity to their preferred cultural amenities in the borough (based on the cluster/epicentre in the borough to selected categories of cultural infrastructure). The project also provides the Ofted's ratings for the schools, as well as the students gender in the school and the type of the school to help families choose the best location based on the educational instituiton. Finally,the project provides 5 closest cultural amenities to selected school, which can also be an imporant factor for families selecting schools and places to live in the borough.

# Preparing the Data

## Data Collection:

To accomplish the task there were several datasets required: 

1) Schools Dataset: schools dataset was aquired from London School Atlas datasets - it includes schools names and their URNs (unique reference numbers), as well as their location based on the longitude and langitude - available at London Datastore - https://data.london.gov.uk/dataset/london-schools-atlas 

2) Ofsted's Ratings Data - data provided from Ofsted available at Gov.UK website - https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/c0c08e6d-c3ef-4408-8193-dcc493b7fa59 

3) Tower Hamlets Wards Geodata: data providing geolocations of wards in Tower Hamlers; data available at https://mapit.mysociety.org/area/2506.html

4) Cultural Infrastructure Data: provided by Greater London Authority (GLA) - dataset that provides coordinates, categories and names of all the cultural infrastructure in London (ex. cinemas, libraries, museums and galleries, dancing studios, etc.); data available at https://data.london.gov.uk/dataset/cultural-infrastructure-map-2023

In [624]:
import pandas as pd
import numpy as np
import xlrd
import folium
import zipfile
import io
import geopandas as gpd
import os
from sklearn.cluster import KMeans

from fastkml import kml

In [625]:
# Schools location data filtered for all the schools in Tower Hamlets only
schools_location_data = pd.read_csv("LondonSchoolsAtlas_Dataset_2015/all_schools_xy.csv")
tower_hamlets_schools_data = schools_location_data.loc[schools_location_data["LA_NAME"] == "Tower Hamlets"]
tower_hamlets_schools_data = tower_hamlets_schools_data[['SCHOOL_NAM', 'TYPE', 'PHASE', 'GENDER', 'WARD_NAME', 'WEBLINK', 'URN', 'POINT_X', 'POINT_Y']]

# Ofsted's Schools Rating Data
quality_of_schools_data = pd.read_csv('quality_of_schools_ofsted.csv')

# Filter for only Tower Hamlets schools
quality_of_schools_data = quality_of_schools_data.loc[quality_of_schools_data['la_name'] == 'Tower Hamlets']


In [626]:
tower_hamlets_schools_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 118 entries, 412 to 4071
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SCHOOL_NAM  118 non-null    object 
 1   TYPE        118 non-null    object 
 2   PHASE       118 non-null    object 
 3   GENDER      118 non-null    object 
 4   WARD_NAME   118 non-null    object 
 5   WEBLINK     118 non-null    object 
 6   URN         118 non-null    int64  
 7   POINT_X     118 non-null    float64
 8   POINT_Y     118 non-null    float64
dtypes: float64(2), int64(1), object(6)
memory usage: 9.2+ KB


In [627]:
quality_of_schools_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 83 entries, 584 to 666
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   time_period                   83 non-null     int64 
 1   time_identifier               83 non-null     object
 2   geographic_level              83 non-null     object
 3   country_code                  83 non-null     object
 4   country_name                  83 non-null     object
 5   region_code                   83 non-null     object
 6   region_name                   83 non-null     object
 7   old_la_code                   83 non-null     int64 
 8   new_la_code                   83 non-null     object
 9   la_name                       83 non-null     object
 10  school_laestab                83 non-null     int64 
 11  school_urn                    83 non-null     int64 
 12  school_name                   83 non-null     object
 13  school_type             

In [628]:
tower_hamlets_schools_data.head()

,SCHOOL_NAM,TYPE,PHASE,GENDER,WARD_NAME,WEBLINK,URN,POINT_X,POINT_Y
412,Ben Jonson Primary School,Community School,Primary,Mixed,St. Dunstan's and Stepney Green,www.benjonson.towerhamlets.sch.uk,100890,-0.038560,51.521041
413,Bonner Primary School,Community School,Primary,Mixed,Mile End and Globe Town,,100891,-0.048668,51.529725
414,Old Palace Primary School,Community School,Primary,Mixed,Bromley-by-Bow,http://www.oldpalaceprimary.co.uk/,100892,-0.014142,51.527636
415,Canon Barnett Primary School,Community School,Primary,Mixed,Spitalfields and Banglatown,,100893,-0.071784,51.516322
416,Cayley Primary School,Community School,Primary,Mixed,St. Dunstan's and Stepney Green,,100894,-0.038386,51.516047


In [629]:
quality_of_schools_data.head()

,time_period,time_identifier,geographic_level,country_code,country_name,region_code,region_name,old_la_code,new_la_code,la_name,...,school_name,school_type,school_main_phase,school_phase,primary_existing_places,primary_new_places,secondary_existing_places,secondary_new_places,ofsted_overall_effectiveness,inspection_Date
584,202122,Academic year,School,E92000001,England,E13000001,Inner London,211,E09000030,Tower Hamlets,...,Ben Jonson Primary School,Community school,Primary,PS,630,0,0,0,Good,01/11/2017
585,202122,Academic year,School,E92000001,England,E13000001,Inner London,211,E09000030,Tower Hamlets,...,Bonner Primary School,Community school,Primary,PS,850,0,0,0,Good,17/10/2019
586,202122,Academic year,School,E92000001,England,E13000001,Inner London,211,E09000030,Tower Hamlets,...,Old Palace Primary School,Community school,Primary,PS,417,0,0,0,Outstanding,07/07/2009
587,202122,Academic year,School,E92000001,England,E13000001,Inner London,211,E09000030,Tower Hamlets,...,Canon Barnett Primary School,Community school,Primary,PS,350,0,0,0,Good,04/10/2013
588,202122,Academic year,School,E92000001,England,E13000001,Inner London,211,E09000030,Tower Hamlets,...,Cayley Primary School,Community school,Primary,PS,630,0,0,0,Good,14/06/2012


In [630]:
tower_hamlets_schools_duplicate_rows = tower_hamlets_schools_data[tower_hamlets_schools_data.duplicated(subset=['SCHOOL_NAM'], keep=False)]
print(tower_hamlets_schools_duplicate_rows)

                         SCHOOL_NAM                      TYPE           PHASE  \
469      St Paul's Way Trust School         Foundation School     All Through   
470                  Phoenix School  Community Special School  Not applicable   
2508     St Paul's Way Trust School         Foundation School     All Through   
2516                 Phoenix School  Community Special School  Not applicable   
3220  River House Montessori School  Other Independent School  Not applicable   
3586                Al-Mizan School  Other Independent School  Not applicable   
3626    Excellence Christian School  Other Independent School  Not applicable   
3746  River House Montessori School  Other Independent School  Not applicable   
3993                Al-Mizan School  Other Independent School  Not applicable   
4023    Excellence Christian School  Other Independent School  Not applicable   

     GENDER            WARD_NAME       WEBLINK     URN   POINT_X    POINT_Y  
469   Mixed        Mile End Ea

In [631]:
counts_tower_hamlets_schools_data = tower_hamlets_schools_data['SCHOOL_NAM'].value_counts()
duplicate_counts = counts_tower_hamlets_schools_data[counts_tower_hamlets_schools_data > 1]
print(duplicate_counts)

SCHOOL_NAM
Phoenix School                   2
St Paul's Way Trust School       2
River House Montessori School    2
Al-Mizan School                  2
Excellence Christian School      2
Name: count, dtype: int64


In [632]:
null_counts_tower_hamlets_schools_data = tower_hamlets_schools_data[['POINT_X', 'POINT_Y']].isnull().sum()
print(null_counts_tower_hamlets_schools_data)

POINT_X    0
POINT_Y    0
dtype: int64


In [633]:
null_count_quality_of_schools_data = quality_of_schools_data['ofsted_overall_effectiveness'].isnull().sum()
print(null_count_quality_of_schools_data)

0


In [634]:
counts_quality_of_schools_data = quality_of_schools_data['school_name'].value_counts()
duplicate_counts_quality_of_schools_data = counts_quality_of_schools_data[counts_quality_of_schools_data > 1]
print(duplicate_counts_quality_of_schools_data)

Series([], Name: count, dtype: int64)


In [635]:
value_counts_quality_of_schools_data = quality_of_schools_data['ofsted_overall_effectiveness'].value_counts(dropna=False)
print(value_counts_quality_of_schools_data)

ofsted_overall_effectiveness
Good                    54
Outstanding             26
Requires improvement     3
Name: count, dtype: int64


# Data Description

There is 118 schools in Tower Hamlets in the London School Atlas dataset (tower_hamlets_schools_data). London School Atlas datset provides information about the school such as name, gender of students, type, link to school's website, as well as the coordinates and the URN (unique reference number) of the school. 

There is data for 83 schools located in Tower Hamlets in the Ofsted's schools ratings dataset (quality_of_schools_data). The most important data from this dataset is Ofted's rating of the school and the URN to map the schools to London School Atlas datset.

# Data Preprocessing and Ethics

The schools locations data and the Ofsted's rating data needed to be merged on the URN (unique reference number) of the schools in London School Atlas dataset - (URN column in tower_hamlets_schools_data and school_urn in quality_of_schools_data).

It's imporant to note that London School Atlas dataset has schools for which there is no Ofted's ratings, therefore the value of for the rating of those schools will not be provided. It would also be significant to raise the question of why some schools are missing the data. There are several reasons for why that could be: there might be missing data in Ofted's datasets or those schools might be new and there are no ratings for them yet.

In London School Atlas there are 5 schools that were repeted twice in the dataset, therefore those duplicated values were removed. 

Total number of schools in London School Atlas is therefore: 113 (no missing values for the coordinates of the schools)
Total number of schools in Ofsted's dataset is: 83 (there were no duplicated values or missing ratings)

In [636]:
# Merge Tower Hamlets schools data with the schools Ofsted's ratings
quality_subset = quality_of_schools_data[['school_urn', 'ofsted_overall_effectiveness']]
merged_df = pd.merge(
    tower_hamlets_schools_data,
    quality_subset,
    left_on='URN',
    right_on='school_urn',
    how='left'
)

# Drop URN columns as we don't need it for user interface or analysis anymore
merged_df = merged_df.drop(columns=['school_urn', 'URN'])

In [637]:
# Check for how many schools we don't have Ofsted's ratings
num_nulls = merged_df['ofsted_overall_effectiveness'].isnull().sum()
print("Number of schools missing Ofted's ratings:", num_nulls)

Number of schools missing Ofted's ratings: 44


# Exploring and Modelling the Data

To visualise the data Tower Hamlet's wards were mapped using Folium library. Then the schools data from London School Atlas were plotted on the map with information on them using the coordinates. Finally, all the cultural infrastructure was mapped using their names and categories.

In [638]:
# Create folium map centred on the London Tower Hamlet location
m = folium.Map(location=(51.49, -0.054), zoom_start=13, tiles='cartodb positron')

In [639]:
# Load the geojson files for Tower Hamlet wards
geojson_dir = "Tower_Hamlets_Wards"

geojson_files = [f for f in os.listdir(geojson_dir) if f.endswith(".geojson")]

name_corrections = {
    "stkatharines and wapping": "St Katharines Docks and Wapping",
    "stdustans": "St Dunstan's"
}

for file_name in geojson_files:
    file_path = os.path.join(geojson_dir, file_name)
 
    display_name = file_name.replace("_", " ").replace("&", " and ").replace(".geojson", "").title()

    display_name = display_name.replace(" And ", " and ")
 
    display_name = name_corrections.get(display_name.lower(), display_name)

    gdf = gpd.read_file(file_path)

    geojson_layer = folium.GeoJson(gdf)
    geojson_layer.add_child(folium.Tooltip(display_name))
    geojson_layer.add_to(m)

folium.LayerControl().add_to(m)

m

In [640]:
# Plot the schools on the map with information on the schools
for idx, row in merged_df.iterrows():
    weblink = row['WEBLINK'] if pd.notnull(row['WEBLINK']) and row['WEBLINK'] != "" else "Not available"
    ofsted = row['ofsted_overall_effectiveness'] if pd.notnull(row['ofsted_overall_effectiveness']) and row['ofsted_overall_effectiveness'] != "" else "Not available"
    
    popup_html = f"""
    <b>{row['SCHOOL_NAM']}</b><br><br>
    Type: {row['TYPE']}<br><br>
    Phase: {row['PHASE']}<br><br>
    Gender: {row['GENDER']}<br><br>
    Ward: {row['WARD_NAME']}<br><br>
    Weblink: <a href="{weblink}" target="_blank">{weblink}</a><br><br>
    Ofsted Overall Effectiveness: {ofsted}
    """
    
    folium.Marker(
        location=[row['POINT_Y'], row['POINT_X']],
        popup=popup_html,
        icon=folium.Icon(prefix="fa", icon="school", color='green')).add_to(m)

m


In [641]:
# Map the cultural infrastructure based in Tower Hamlets
data_dir = "cultural_infrastructure"
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

icon_mapping = {
    "Cinemas": "film",
    "Theatre": "ticket",
    "Creative coworking desk space": "paintbrush",
    "Artists workspaces": "paintbrush",
    "Dance performance venues": "music",
    "Dance rehearsal studios": "music",
    "Libraries": "book",
    "Museums and public galleries": "building-columns",
    "Jewellery design": "gem",
    "Fashion and design": "bag-shopping",
    "Lgbt venues": "flag",
    "Textile design": "scissors",
    "Making and manufacturing": "screwdriver-wrench",
    "Prop and costume making": "wand-magic-sparkles",
    "Set and exhibition building": "paintbrush",
    "Music Venues": "music",  # Updated key for Music Venues
    "Theatre rehearsal studios": "masks-theater",
    "Makerspaces": "craft",
    "Creative workspaces": "ruler",
    "Commercial galleries": "building-columns",
    "Arts centres": "brush",
}

for file_name in csv_files:
    file_path = os.path.join(data_dir, file_name)
    category = file_name.replace("CIM 2023 ", "").replace(" (Nov 2023)", "").replace(".csv", "")
    if category == "CIM 2024 Music_Venues_All":
        category = "Music Venues"
    df = pd.read_csv(file_path)
  
    if {"longitude", "latitude", "name", "borough_name"}.issubset(df.columns):
        df = df[df["borough_name"].str.contains("tower hamlets", case=False, na=False)]
        if not df.empty:
            icon_name = icon_mapping.get(category, "info-sign")
            for _, row in df.iterrows():
                folium.Marker(
                    location=[row["latitude"], row["longitude"]],
                    popup=f"<b>{row['name']}</b><br><i>{category}</i>",
                    icon=folium.Icon(icon=icon_name, prefix="fa", color="blue")).add_to(m)

m


# Modelling

For the modelling part the K-Means Clustering model imported from sklearn was used to find the epicentres in Tower Hamlet borough to specific categories of cultural amenities. By choosing particular categies of cultural infrastructures(e.g. ['libraries', 'museums']) we can find in what part of the boroughs you need to be located to be closest to as many of the cultural places from the categories selected as possible. This could be used to assess a preferred location to live. 

In the application, the user has a chance to select preferred categories of cultural infrastructure as well as the number of epicentres/clusters. For example, by choosing 3 clusters, there will be three red markers on the map showing three places in which you can be located to be closest to as many cultural places as possible.

The K-Means clustering uses Euclidean Distance meatric to divide all the cultural venues from the categories selected and divide them into the number of clusters selected based, choosing a cluster centre that is the point closest to all the venues in that particular cluster. 

This image visualises how the cluster centres are mapped: https://i0.wp.com/neptune.ai/wp-content/uploads/2024/04/k-means-clustering-3.jpg?w=1200&ssl=1 

This image visualises colors of the clusters and how do they split the venues: https://i0.wp.com/neptune.ai/wp-content/uploads/2024/04/k-means-clustering-4.jpg?w=1200&ssl=1



In [642]:
# ---------------------------
# Clustering on Cultural Infrastructure
# ---------------------------
# Explanation:
# This block first combines cultural infrastructure data from all CSV files into a single DataFrame.
# It then applies KMeans clustering on the latitude and longitude columns.
# KMeans clustering partitions the data points into 'n_clusters' groups by minimizing the distance 
# between each data point and its respective cluster centre. The computed cluster centres are then 
# added to the folium map as markers, and each cultural point is plotted with a circle marker colored 
# according to its cluster label. This visualization helps identify areas with a high concentration 
# of cultural places.
#
# In other words, clusters shows the epicentres of the cultural infrastructures. It could be interpreted
# as the optimal place to live if the proximity to specific cultural places is the factor to base the 
# living location.
# ---------------------------

cultural_list = []
for file_name in csv_files:
    file_path = os.path.join(data_dir, file_name)
    category = file_name.replace("CIM 2023 ", "").replace(" (Nov 2023)", "").replace(".csv", "")
    df = pd.read_csv(file_path)
    if {"longitude", "latitude", "name", "borough_name"}.issubset(df.columns):
        df = df[df["borough_name"].str.contains("tower hamlets", case=False, na=False)]
        if not df.empty:
            df['category'] = category
            cultural_list.append(df)
if cultural_list:
    cultural_all = pd.concat(cultural_list, ignore_index=True)
else:
    cultural_all = pd.DataFrame()

if not cultural_all.empty:
    X = cultural_all[['latitude', 'longitude']]
    n_clusters = 3 
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(X)
    cultural_all['cluster'] = cluster_labels

    colors = ['purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']
    
    centers = kmeans.cluster_centers_
    for i, center in enumerate(centers):
        cluster_color = colors[i % len(colors)]
        folium.Marker(
            location=[center[0], center[1]],
            popup=f"Cluster {i+1} Centre",
            icon=folium.Icon(color=cluster_color, icon="star", prefix="fa")).add_to(m)
    
    for idx, row in cultural_all.iterrows():
        cluster_color = colors[row['cluster'] % len(colors)]
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4,
            color=cluster_color,
            fill=True,
            fill_color=cluster_color,
            fill_opacity=0.6,
            popup=f"{row['name']} (Cluster {row['cluster']+1})").add_to(m)
else:
    print("No cultural infrastructure data available for clustering.")

m

# Modelling 

Simple modelling technique using Euclidean distance was also used to locate 5 closest cultural places to selected school. This is a simple technique caluclating the distance using Euclidean formula and sorting based on the closest distance to the selected school. 

In the application, the user has a chance to click on the school and beneath the map there will appear a list of 5 closest cultural locations to the selected school. 

In [643]:
# ---------------------------
# Show 5 Closest Cultural Infrastructures for a Given School
# ---------------------------
# Explanation:
# This code block defines a helper function, get_closest_cultural_infra, which calculates the Euclidean
# distance between a specified school's location and every cultural infrastructure point in the combined
# dataset (cultural_all). The Euclidean distance (sqrt((Δlat)² + (Δlon)²)) is used here for simplicity.
# The function then sorts the cultural points by distance and returns the top 5 closest infrastructures.
# Finally, the code selects an example school (here, the first school in merged_df) and displays a table
# showing the 5 closest cultural infrastructures to that school.
#
# This is an example of how Streamlit app shows 5 closest cultural infrastructures for each school selected 
# by the user. 
# ---------------------------

def get_closest_cultural_infra(school_lat, school_lon, cultural_df, n=5):
    df = cultural_df.copy()
    df['distance'] = ((df['latitude'] - school_lat)**2 + (df['longitude'] - school_lon)**2)**0.5
    return df.nsmallest(n, 'distance')[["name", "category", "latitude", "longitude", "distance"]]

example_school = merged_df.iloc[0]
school_lat = example_school['POINT_Y']
school_lon = example_school['POINT_X']

if not cultural_all.empty:
    closest_infra = get_closest_cultural_infra(school_lat, school_lon, cultural_all, n=5)
    print(f"5 Closest Cultural Infrastructures to {example_school['SCHOOL_NAM']}:")
    display(closest_infra)
else:
    print("No cultural infrastructure data available for computing closest infrastructures.")


5 Closest Cultural Infrastructures to Ben Jonson Primary School:


,name,category,latitude,longitude,distance
65,Local History Library,Libraries,51.523309,-0.040758,0.003158
79,Copperfield Road,Artists workspaces,51.519079,-0.035805,0.003382
208,Haileybury Centre,Dance rehearsal studios,51.517600,-0.042499,0.005230
98,Ragged School Museum,Museums and public galleries,51.518152,-0.034074,0.005336
39,Half Moon Young People's Theatre,Theatre,51.514239,-0.042221,0.007724


# Models Evaluation

It's imporant to note limitations of the K-Means Clustering model as well as the Euclidean Distance model.

The clustering model suggests optimal place to be closest to the cultural infrastructure selected, however it fails to recognise physical and geographical constraints. For example, the cluster might suggest the most optimal place in the middle of the train station, which is unrealistic. 

For Euclidean Distance model, the physical and geographical constraints are also not taked into consideration as it takes the distance in the straight line, not considering that there are roads, houses, and other infrastructure on the way. It might turn out that the acutal distance is further than calculated using Euclidean formula. 

# Deploying Findings via Streamlit Application

The findings, including Tower Hamlet wards and schools, as well as cultural infrastructure visualisations and clustering modelling that shows cultural infrastructure epicentres, were presented in a Streamlit application. Users can interact with the applicatiton where the main part it the London map and the sidebar that allows to select cultural infrastructure categories and select the numbers of epicentres on the cultural infrastructure categories they are interested in. Those epicentres will show what cultural places are part of the cluster/epicentre using the same colour that the epicentre marker is show in. 

The app can benefit local residents and any stakeholders interested in the opporunities and quality of education and cultural infrastructure in the Tower Hamlets through visualising all the cultural amentities and schools, providing the quality of the school based on Ofsted's rating as well as closest cultural amenities to the school. 

Application is simple, intuitive and interactive making the datasets and data science techniques accessible to non-technical, as well as technical users. 

# Conclusion and Further Development

The key findings of the app are the geo-location of all the cultural amenities and schools in the Tower Hamlet borough splitted into wards. Another imporant finding is the clustering that shows the geo-location of the epicetres in the Tower Hamlets borough which shows what parts of the borough are the most connected with as many cultural amenities from selected categories as possible. This might impact the community through showcasing schools that are most connected to cultural amenities, as well as quantity of cultural infrastructure. The clustering/epicentres show what might be the best location to be located to be connected with cultural amentities. This might provide useful when choosing a place of residence or office in the borough. 

The future development of the project would include analysis of how each school utilises the cultural amentities close to it - for example are school organising trips at events at the cultural places, are they partnering with those places, are they actively trying to engage students and help them become curious and imagine what they can do with their education. This analysis will be achieved through scraping the data from the web about each school and their events. 

Additional further work will include extending the boroughs to other East London boroughs and eventually the whole London. At that point, the project would also allow for comparasions between boroughs. Including additional datasets, like the crime dataset, median income of the wards, air quality and green spaces, would provide a comprehensive overview of the boroughs and the wards on a large scale. This is the direction in which this project will go in the future work and development.